In [4]:
import os
import shutil
import random

def sample_and_move_parallel_files(source_folder, destination_folder, sample_ratio):
    """
    'images'와 'labels' 폴더가 병렬로 구성된 구조에서 파일을 찾아
    지정된 비율만큼 랜덤 샘플링하여 타겟 폴더로 이동시킵니다.
    이때 하위 폴더 구조를 그대로 유지합니다.

    Args:
        source_folder (str): 'images', 'labels' 폴더가 있는 최상위 소스 폴더.
        destination_folder (str): 파일을 옮겨놓을 타겟 폴더.
        sample_ratio (float): 옮길 파일의 비율 (0.0 ~ 1.0 사이).
    """
    # 소스 폴더 내의 images와 labels 경로 설정
    image_source_root = os.path.join(source_folder, 'images')
    label_source_root = os.path.join(source_folder, 'labels')

    # 타겟 폴더 내의 images와 labels 경로 설정 및 생성
    image_dest_root = os.path.join(destination_folder, 'images')
    label_dest_root = os.path.join(destination_folder, 'labels')
    os.makedirs(image_dest_root, exist_ok=True)
    os.makedirs(label_dest_root, exist_ok=True)

    # 소스 경로 존재 여부 확인
    if not os.path.isdir(image_source_root) or not os.path.isdir(label_source_root):
        print(f"오류: 소스 폴더 '{source_folder}' 내에 'images'와 'labels' 폴더가 모두 존재해야 합니다.")
        return

    # 찾을 파일 이름 패턴
    patterns_to_find = ('_low_light_l', '_low_light_h')

    # 1. 이동할 파일 후보 목록 생성 (이미지 기준)
    file_pairs = []
    print("📂 'images' 폴더를 기준으로 파일 검색 중...")
    for dirpath, _, filenames in os.walk(image_source_root):
        for filename in filenames:
            base_name, ext = os.path.splitext(filename)
            
            # 이미지 파일 확장자 및 지정된 패턴 확인
            if ext.lower() in ('.jpg', '.png') and any(base_name.endswith(p) for p in patterns_to_find):
                image_path = os.path.join(dirpath, filename)
                
                # 대응하는 라벨 파일 경로 추정
                # 1. images 루트로부터의 상대 경로 계산
                relative_path = os.path.relpath(image_path, image_source_root)
                # 2. 확장자를 .txt로 변경
                label_relative_path = os.path.splitext(relative_path)[0] + '.txt'
                # 3. labels 루트에 상대 경로를 붙여 최종 경로 생성
                label_path = os.path.join(label_source_root, label_relative_path)

                # 라벨 파일이 실제로 존재하는지 확인
                if os.path.exists(label_path):
                    file_pairs.append({'image': image_path, 'label': label_path})
    
    if not file_pairs:
        print("이동할 이미지-라벨 쌍을 찾지 못했습니다.")
        return

    print(f"✅ 총 {len(file_pairs)}개의 이미지-라벨 쌍을 찾았습니다.")

    # 2. 지정된 비율만큼 랜덤 샘플링
    number_to_sample = int(len(file_pairs) * sample_ratio)
    sampled_pairs = random.sample(file_pairs, number_to_sample)
    
    print(f"🎯 샘플링 비율: {sample_ratio * 100:.1f}%")
    print(f"총 {len(sampled_pairs)}개의 쌍을 이동합니다.")
    print("-" * 30)

    # 3. 샘플링된 파일들 이동 (하위 폴더 구조 유지)
    moved_count = 0
    skipped_count = 0

    for pair in sampled_pairs:
        image_source_path = pair['image']
        label_source_path = pair['label']
        
        # 이동 후에도 유지할 상대 경로 계산
        relative_path_img = os.path.relpath(image_source_path, image_source_root)
        relative_path_label = os.path.relpath(label_source_path, label_source_root)
        
        # 최종 목적지 경로 계산
        image_dest_path = os.path.join(image_dest_root, relative_path_img)
        label_dest_path = os.path.join(label_dest_root, relative_path_label)

        # 중복 파일 확인
        if os.path.exists(image_dest_path):
            print(f"⚠️ 건너뛰기 (파일이 이미 존재함): {os.path.basename(image_dest_path)}")
            skipped_count += 1
            continue
        
        try:
            # 이동할 경로의 폴더가 없으면 생성
            os.makedirs(os.path.dirname(image_dest_path), exist_ok=True)
            os.makedirs(os.path.dirname(label_dest_path), exist_ok=True)

            # 파일 이동
            shutil.move(image_source_path, image_dest_path)
            shutil.move(label_source_path, label_dest_path)
            print(f"🚀 이동 완료: {os.path.basename(image_dest_path)}")
            moved_count += 1
        except Exception as e:
            print(f"❌ 오류: {os.path.basename(image_source_path)} 이동 중 오류 발생 - {e}")

    print("-" * 30)
    print("✅ 작업 완료!")
    print(f"총 이동된 파일 쌍: {moved_count}개")
    print(f"건너뛴 파일 쌍 (중복): {skipped_count}개")

if __name__ == '__main__':
    source = input("파일을 검색할 상위 폴더 경로를 입력하세요 ('images', 'labels' 폴더가 있는 곳): ").strip()
    destination = input("정리된 파일을 저장할 타겟 폴더 경로를 입력하세요: ").strip()
    
    while True:
        try:
            ratio_input = input("이동할 데이터의 비율을 입력하세요 (예: 70% -> 0.7): ").strip()
            ratio = float(ratio_input)
            if 0.0 <= ratio <= 1.0:
                break
            else:
                print("오류: 비율은 0.0과 1.0 사이의 숫자여야 합니다.")
        except ValueError:
            print("오류: 올바른 숫자를 입력해주세요.")

    if not os.path.isdir(source):
        print(f"오류: 소스 폴더 '{source}'를 찾을 수 없습니다.")
    elif not os.path.isdir(destination):
        print(f"오류: 타겟 폴더 '{destination}'를 찾을 수 없습니다. 먼저 폴더를 생성해주세요.")
    else:
        sample_and_move_parallel_files(source, destination, ratio)

📂 'images' 폴더를 기준으로 파일 검색 중...
✅ 총 33330개의 이미지-라벨 쌍을 찾았습니다.
🎯 샘플링 비율: 50.0%
총 16665개의 쌍을 이동합니다.
------------------------------
🚀 이동 완료: S63_DATA2_ON_L1_D2023-10-18-10-40_001_001215_low_light_l.jpg
🚀 이동 완료: S63_DATA2_FN_L1_D2023-08-25-14-30_001_000197_low_light_h.jpg
🚀 이동 완료: S63_DATA2_CN_L1_D2023-10-26-10-50_001_000166_low_light_h.jpg
🚀 이동 완료: S63_DATA2_FN_L1_D2023-09-07-14-14_001_000488_low_light_l.jpg
🚀 이동 완료: S63_DATA2_FP_L1_D2023-08-25-13-35_001_000017_low_light_h.jpg
🚀 이동 완료: S63_DATA2_FP_L1_D2023-10-26-13-37_006_000385_low_light_l.jpg
🚀 이동 완료: S63_DATA2_CN_L1_D2023-09-14-10-30_001_000684_low_light_l.jpg
🚀 이동 완료: S63_DATA2_CP_L1_D2023-09-14-10-50_001_000174_low_light_h.jpg
🚀 이동 완료: S63_DATA2_FP_L1_D2023-09-07-13-53_001_000271_low_light_l.jpg
🚀 이동 완료: S63_DATA2_CN_L1_D2023-10-26-10-45_001_000215_low_light_l.jpg
🚀 이동 완료: S63_DATA2_SP_L1_D2023-08-25-10-38_001_006028_low_light_h.jpg
🚀 이동 완료: S63_DATA2_SP_L1_D2023-08-25-10-11_001_001283_low_light_h.jpg
🚀 이동 완료: S63_DATA2_OP_L1_D2023-10